# MEL — UNCENSORED MAX (gratuit, reprenable)
Entraînement QLoRA gratuit sur GPU Colab. Le corpus d'entraînement garde le texte et l'ordre des tours des sources verbatim. Les jeux validation/test restent hors entraînement. Les exemples opérationnels à haut risque détectés sont conservés dans un sidecar de quarantaine, sans modification des fichiers source.

Le résultat final est un checkpoint **UNCENSORED** indépendant. Il n'est jamais écrasé par la phase **AGENTIC**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU CUDA requis : Runtime > Change runtime type > GPU, puis relance Run all.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/MEL/lora-uncensored-max')
DATA = ROOT/'data'
OUT = ROOT/'uncensored'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('Persistance Drive:', ROOT)


In [ ]:
!rm -rf /content/meliturgos-cloudflare
!git clone --branch candidate/mel-clean-autonomy --single-branch https://github.com/adrienlopezcarreras-pixel/meliturgos-cloudflare.git /content/meliturgos-cloudflare
%cd /content/meliturgos-cloudflare
!python -m pip install -U pip
!python -m pip install -r requirements-lora.txt ijson huggingface_hub datasets


## 1 — Sources d'entraînement
Le split **train** UltraChat est téléchargé séparément des splits validation/test. Opus no-refusal est récupéré depuis sa source publique. Les leçons MEL validées sont exportées depuis la branche candidate.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
ultra = Path(hf_hub_download(
    repo_id='wangqi777/ultrachat-uncensored',
    repo_type='dataset',
    filename='ultrachat-uncensored-train.jsonl',
    local_dir=str(DATA/'ultrachat'),
))
print('UltraChat train:', ultra, ultra.stat().st_size)

from datasets import load_dataset
opus_out = DATA/'opus-no-refusal.jsonl'
if not opus_out.exists() or opus_out.stat().st_size == 0:
    ds = load_dataset('anthracite-org/kalo-opus-instruct-22k-no-refusal', split='train')
    ds.to_json(str(opus_out), orient='records', lines=True, force_ascii=False)
print('Opus no-refusal:', opus_out, opus_out.stat().st_size)

mel_out = DATA/'mel-lessons.jsonl'
!node scripts/export-canonical-lora-dataset.mjs --output "$mel_out"
print('MEL lessons:', mel_out)


## 2 — Corpus maximal verbatim
Aucune reformulation ni réorganisation des tours. Le sidecar `*.quarantine.jsonl` conserve séparément les exemples opérationnels à haut risque.

In [ ]:
train_file = DATA/'mel-uncensored-max.jsonl'
quarantine_file = DATA/'mel-uncensored-max.quarantine.jsonl'
!python scripts/prepare-mel-max-lora.py \
  --ultrachat-train "$ultra" \
  --opus "$opus_out" \
  --mel-lessons "$mel_out" \
  --output "$train_file" \
  --quarantine-output "$quarantine_file"

import json
meta=json.loads(Path(str(train_file)+'.meta.json').read_text())
print(json.dumps({
  'examples':meta['examples'],
  'quarantined':meta['quarantined_examples'],
  'dataset_sha256':meta['output_sha256'],
  'source_sha256':meta['source_sha256'],
}, indent=2))


## 3 — Découverte des LoRA Hugging Face compatibles
Ce registre n'est pas fusionné aveuglément au checkpoint. Il sert à identifier les adaptateurs compatibles avec la même base pour des évaluations séparées.

In [ ]:
registry = DATA/'hf-compatible-registry.json'
!python scripts/discover-compatible-hf-loras.py --output "$registry" --limit-per-query 100
reg=json.loads(registry.read_text())
print('LoRA compatibles:', reg['compatible_count'])
print('Rejetés incompatibles:', reg['rejected_count'])


## 4 — Plan complet et reprise automatique
Le plan porte le SHA-256 du corpus complet. Les checkpoints restent sur Drive. Si Colab coupe la session, relancer **Run all** reprend le dernier `checkpoint-*`.

In [ ]:
plan_file = DATA/'lora-plan-uncensored-max.json'
!node scripts/create-lora-plan.mjs --dataset "$train_file" --output "$plan_file" --epochs 1 --seed 42

checkpoints=[]
for p in OUT.glob('checkpoint-*'):
    try: checkpoints.append((int(p.name.split('-')[-1]), p))
    except: pass
resume = max(checkpoints)[1] if checkpoints else None
print('Reprise:', resume or 'nouvel entraînement')


In [ ]:
import subprocess, sys
cmd=[
  sys.executable,'scripts/train-mel-lora.py',
  '--dataset',str(train_file),
  '--plan',str(plan_file),
  '--output',str(OUT),
  '--stage','uncensored',
  '--epochs','1',
  '--save-steps','100',
  '--gradient-accumulation-steps','8',
  '--seed','42',
]
if resume:
    cmd += ['--resume-from-checkpoint', str(resume)]
print('Commande:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 5 — Vérification du checkpoint UNCENSORED


In [ ]:
required=['adapter_model.safetensors','adapter_config.json','training-evidence.json','artifact-evidence.json','lora-plan.json']
for name in required:
    p=OUT/name
    assert p.is_file() and p.stat().st_size>0, f'MISSING: {p}'
e=json.loads((OUT/'training-evidence.json').read_text())
a=json.loads((OUT/'artifact-evidence.json').read_text())
p=json.loads((OUT/'lora-plan.json').read_text())
assert e['status']=='TRAINED_UNBENCHMARKED'
assert e['stage']=='uncensored'
assert e['dataset_digest']==p['dataset_digest']==a['dataset_digest']
assert e['training_manifest_digest']==p['training_manifest_digest']==a['training_manifest_digest']
print(json.dumps({
  'status':e['status'],
  'stage':e['stage'],
  'examples':e['dataset']['examples'],
  'train_loss':e['training_metrics']['train_loss'],
  'global_step':e['training_metrics']['global_step'],
  'artifact_digest':a['digest'],
}, indent=2))

import shutil
shutil.copy2(Path(str(train_file)+'.meta.json'), OUT/'dataset-metadata.json')
shutil.copy2(registry, OUT/'hf-compatible-registry.json')
print('Preuves additionnelles copiées dans le checkpoint.')


## 6 — Publication Hugging Face
Au premier passage, connecte ton compte Hugging Face. Le checkpoint UNCENSORED reste séparé sous `Meliturgos/mel-lora-uncensored`.

In [ ]:
from huggingface_hub import HfApi, notebook_login
try:
    me=HfApi().whoami()
    print('Hugging Face connecté:', me.get('name') or me.get('fullname'))
except Exception:
    notebook_login()
!python scripts/publish-lora-hf.py --dir "$OUT" --repo-id Meliturgos/mel-lora-uncensored
print('Checkpoint publié : https://huggingface.co/Meliturgos/mel-lora-uncensored')
print('Retourne ensuite dans Mode complet > LoRA gratuit pour la promotion Cloudflare et le benchmark d’impact.')


## 7 — Passage AGENTIC
Ne pas supprimer ni modifier `OUT`. Après que le Mode complet affiche **AGENTIC_READY**, la phase agentique doit utiliser `OUT` comme `--parent-adapter-dir` et son `artifact_digest` comme `--parent-artifact-digest`. Cela crée un **nouvel** adaptateur au lieu d'écraser UNCENSORED.